# AlexNet CNN encoder — trained from scratch on the augmented dataset

Trains an **AlexNet** encoder (`src/models/encoders/alexnet.py`) from random weights for
200-class bird classification on CUB-200-2011, using the **originals + Task 1.1 augmented copies**
from `src/data` (`data.use_offline_augmented=true`).

All logic lives in `src/` (same data split, normalisation, training loop, evaluation and Grad-CAM as the
from-scratch ResNet); this notebook only configures and runs it. Settings are in `configs/cls_alexnet.yaml`.

**Architecture** (input 224×224):

| layer | output | notes |
|---|---|---|
| conv1 11×11/4, 96 | 55×55 | `c1` |
| maxpool 3/2 → conv2 5×5, 256 | 27×27 | `c2` |
| maxpool 3/2 → conv3 3×3, 384 | 13×13 | `c3` |
| conv4 3×3, 384 | 13×13 | `c4` |
| conv5 3×3, 256 | 13×13 | `c5`, Grad-CAM layer |
| maxpool 3/2 → FC6 4096 → FC7 4096 (dropout 0.5) | 4096 | encoder output (`pooled`) |
| FC8 200 | 200 | classifier head |

BatchNorm replaces the original Local Response Normalisation so the network trains well from scratch.

**Where to run.** A GPU is strongly recommended (~1–2 min/epoch on a T4 vs ~11 min on CPU).

**Secrets.** `GITHUB_TOKEN` is only needed while the repo is private and `WANDB_API_KEY` is optional (without it,
tracking is disabled). They are never stored in the repository: `.env` is git-ignored, so each environment gets them
from its own place, as described below.

- **VS Code with a Colab kernel**: the notebook runs on the Colab server, which cannot see your local files. Connect
  to a T4 GPU, then in the Explorer right-click `.env` → *Upload to Colab*, and run the notebook. New Colab servers
  start empty, so upload it again after getting a new one. The notebook clones the `AlexNet` branch, so the branch
  must be pushed to GitHub. Results are written to `/content/outputs` and zipped into
  `/content/alexnet_results.zip` at the end; `.env` is not included. Download them before removing the server.
- **Google Colab in the browser** (T4): *Runtime → Change runtime type → T4 GPU*. Add the Colab secrets
  `GITHUB_TOKEN` and `WANDB_API_KEY`, or the notebook asks for them in a hidden input box. Results are also copied to
  Google Drive (`MyDrive/alexnet_outputs`) when Drive can be mounted.
- **Kaggle**: enable *GPU* and *Internet*, add the secrets `GITHUB_TOKEN` (and `WANDB_API_KEY`), then *Save & Run All*.
  Do not put a `.env` in `/kaggle/working`: that folder is saved as the notebook output.
- **Locally on your own GPU**: run it from the `notebooks/` folder with a CUDA build of PyTorch. Secrets are read
  from the `.env` file in the repo root.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

ON_KAGGLE = os.path.exists("/kaggle/working")
try:
    import google.colab  # noqa: F401
    ON_COLAB = True
except ImportError:
    ON_COLAB = False


def load_env():
    # Load KEY=value lines from a .env kept outside git; values already in the environment win.
    # Looks next to the notebook / repo root, then in the Colab server's /content (VS Code: right-click
    # .env > Upload to Colab), then in Google Drive if it is already mounted
    folders = (Path.cwd(), Path.cwd().parent, Path("/content"), Path("/content/drive/MyDrive"))
    env_file = next((f / ".env" for f in folders if (f / ".env").exists()), None)
    if env_file is None:
        print("No .env file found" + (" (VS Code: right-click .env > Upload to Colab, then rerun)" if ON_COLAB else ""))
        return
    print("Secrets loaded from", env_file)
    for line in env_file.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if line and not line.startswith("#") and "=" in line:
            key, value = line.split("=", 1)
            os.environ.setdefault(key.strip(), value.strip().strip("'\""))


def get_secret(name):
    # Read a secret from the .env file, then Kaggle / Colab; None when it is not set
    if name in os.environ:
        return os.environ[name]
    try:
        if ON_KAGGLE:
            from kaggle_secrets import UserSecretsClient
            return UserSecretsClient().get_secret(name)
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return None


def ask_secret(name, purpose):
    # Colab only, when there is no .env or Colab secret: the typed value is hidden and never saved
    if not ON_COLAB:
        return None
    from getpass import getpass
    print(f"Waiting for {name} in the input box (top of the window in VS Code)")
    return getpass(f"{name} ({purpose}, leave blank to skip): ").strip() or None


load_env()
WANDB_API_KEY = get_secret("WANDB_API_KEY") or ask_secret("WANDB_API_KEY", "experiment tracking")
if WANDB_API_KEY:
    os.environ["WANDB_API_KEY"] = WANDB_API_KEY
else:
    print("No WANDB_API_KEY secret: tracking disabled")

if ON_KAGGLE or ON_COLAB:
    # A token is only needed while the repo is private
    GITHUB_TOKEN = get_secret("GITHUB_TOKEN") or ask_secret("GITHUB_TOKEN", "only while the repo is private")
    credentials = f"{GITHUB_TOKEN}@" if GITHUB_TOKEN else ""

    REPO = "omarsaher/image-classfication-and-caption"
    BRANCH = "AlexNet"
    WORKDIR = "/tmp/repo" if ON_KAGGLE else "/content/repo"                          # code + data
    OUTPUTS = "/kaggle/working/outputs" if ON_KAGGLE else "/content/outputs"         # results

    # Clone the repo; the token is removed from the remote URL right after
    if not os.path.exists(WORKDIR):
        result = subprocess.run(
            ["git", "clone", "--depth", "1", "-b", BRANCH,
             f"https://{credentials}github.com/{REPO}.git", WORKDIR],
            capture_output=True, text=True, env={**os.environ, "GIT_TERMINAL_PROMPT": "0"},
        )
        print("clone exit code:", result.returncode)
        stderr = result.stderr.replace(GITHUB_TOKEN, "***") if GITHUB_TOKEN else result.stderr
        print(stderr[-2000:])
        if result.returncode != 0:
            raise RuntimeError(f"Could not clone branch '{BRANCH}' of {REPO}. Check that the branch is pushed "
                               "and, while the repo is private, that GITHUB_TOKEN can access it.")
        subprocess.run(["git", "-C", WORKDIR, "remote", "set-url", "origin",
                        f"https://github.com/{REPO}.git"], check=True)
    os.environ.pop("GITHUB_TOKEN", None)         # training does not need it
    os.makedirs(OUTPUTS, exist_ok=True)
    if not os.path.exists(f"{WORKDIR}/outputs"):
        os.symlink(OUTPUTS, f"{WORKDIR}/outputs")
    os.chdir(WORKDIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)               # all project paths are relative to the repo root

if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))

import torch

print("repo root:", Path.cwd())
print("torch", torch.__version__, "| device:",
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

Secrets loaded from /content/.env
clone exit code: 128
Cloning into '/content/repo'...
remote: Write access to repository not granted.
fatal: unable to access 'https://github.com/omarsaher/image-classfication-and-caption.git/': The requested URL returned error: 403



RuntimeError: Could not clone branch 'AlexNet' of omarsaher/image-classfication-and-caption. Check that the branch is pushed and, while the repo is private, that GITHUB_TOKEN can access it.

## 1. Configuration

`QUICK_CHECK = True` runs 1 epoch on a few batches to check that everything works before the full run.
Tracking uses Weights & Biases when a `WANDB_API_KEY` is available, otherwise only local files are written.

In [ ]:
from src.utils.config import load_config

QUICK_CHECK = False

overrides = [
    "data.use_offline_augmented=true",       # train on originals + augmented copies
]
if not torch.cuda.is_available():
    overrides += ["data.num_workers=0"]      # avoids worker start-up cost / Windows issues on CPU
elif ON_COLAB:
    overrides += [f"data.num_workers={os.cpu_count()}"]   # Colab has only 2 CPU cores
if not os.environ.get("WANDB_API_KEY"):
    overrides += ["tracking.backend=none"]
if QUICK_CHECK:
    overrides += ["experiment.name=alexnet_smoke", "train.epochs=1", "train.warmup_epochs=0",
                  "train.max_batches=3", "data.batch_size=16"]

cfg = load_config("configs/cls_alexnet.yaml", overrides)
RUN_DIR = Path(cfg["experiment"]["output_dir"]) / cfg["experiment"]["name"]
print("run:", cfg["experiment"]["name"], "->", RUN_DIR)
print("model:", cfg["model"])
print("train:", cfg["train"])

## 2. Data — originals + augmented copies

In [ ]:
from collections import Counter

import matplotlib.pyplot as plt
from PIL import Image

from src.data.loaders import load_records

train_records, val_records, class_names = load_records(cfg["data"], cfg["experiment"]["seed"])
print(f"{len(class_names)} classes")
print("train images:", dict(Counter(r["image_type"] for r in train_records)), "| total", len(train_records))
print("val images:  ", dict(Counter(r["image_type"] for r in val_records)), "| total", len(val_records))

# One training original next to its augmented copies
by_id = {}
for r in train_records:
    by_id.setdefault(r["original_id"], []).append(r)
group = next(g for g in by_id.values() if len(g) >= 3)
group = sorted(group, key=lambda r: r["image_type"] != "original")[:5]
fig, axes = plt.subplots(1, len(group), figsize=(3 * len(group), 3))
for ax, r in zip(axes, group):
    ax.imshow(Image.open(r["path"]).convert("RGB"))
    ax.set_title(f"{r['image_type']}\n{r['class_name']}", fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 3. Model

In [ ]:
from src.models.registry import build_classifier, count_parameters

model = build_classifier(cfg["model"])
print(model)
print(f"\nparameters: {count_parameters(model):,}")

with torch.no_grad():
    features = model.encoder.forward_features(torch.randn(1, 3, 224, 224))
for name, tensor in features.items():
    print(f"{name:>6}: {tuple(tensor.shape)}")
del model

## 4. Training

AdamW (no weight decay on biases/BatchNorm), linear warmup + cosine decay, label smoothing, mixed precision on GPU.
The best checkpoint by validation accuracy is saved to `best.pt`; training stops early after
`early_stopping_patience` epochs without improvement.

In [ ]:
from src.training.train_classifier import train_classifier

result = train_classifier(cfg)
print(f"best val acc {result.best_val_acc:.4f} at epoch {result.best_epoch} "
      f"({result.epochs_run} epochs, {result.elapsed_sec / 60:.1f} min)")

In [ ]:
import json

from IPython.display import Image as ShowImage, display

display(ShowImage(filename=str(RUN_DIR / "curves.png")))

with open(RUN_DIR / "history.json", "r", encoding="utf-8") as f:
    history = json.load(f)
best = max(history, key=lambda r: r["val_acc"])
print(f"best epoch {best['epoch']}: train acc {best['train_acc']:.4f} | val acc {best['val_acc']:.4f} "
      f"| val loss {best['val_loss']:.4f}")

## 5. Test-set evaluation (5,794 images)

The CUB test split is downloaded from Hugging Face (`Multimodal-Fatima/CUB_test`) the first time it is used.
With `QUICK_CHECK` the validation split is used instead.

In [ ]:
from src.evaluation.evaluate import evaluate_checkpoint

CHECKPOINT = RUN_DIR / "best.pt"
eval_overrides = [o for o in overrides if o.startswith(("data.num_workers", "tracking."))]
split = "val" if QUICK_CHECK else "test"

report = evaluate_checkpoint(str(CHECKPOINT), split, eval_overrides)
display(ShowImage(filename=str(RUN_DIR / f"eval_{split}" / "confusion_matrix.png"), width=600))
print("worst classes:", report["worst_classes"][:5])

## 6. Grad-CAM — where the encoder looks (conv5)

In [ ]:
from src.visualization.gradcam import run_gradcam

figures = run_gradcam(str(CHECKPOINT), split, n_correct=8, n_wrong=8, overrides=eval_overrides)
for kind, path in figures.items():
    print(kind)
    display(ShowImage(filename=str(path)))

## 7. Comparison with the from-scratch ResNet (if its run is available)

In [ ]:
from src.visualization.plots import plot_run_comparison

runs = {"AlexNet (+ augmented)": RUN_DIR,
        "ResNet (+ augmented)": Path(cfg["experiment"]["output_dir"]) / "scratch_B_augmented"}
histories = {}
for name, run_dir in runs.items():
    if (run_dir / "history.json").exists():
        with open(run_dir / "history.json", "r", encoding="utf-8") as f:
            histories[name] = json.load(f)
if len(histories) > 1:
    path = plot_run_comparison(histories, RUN_DIR / "comparison_resnet.png")
    display(ShowImage(filename=str(path)))
else:
    print("ResNet run not found; train it with the README commands to compare.")

## 8. Save the results

In [ ]:
import shutil

if ON_KAGGLE:
    shutil.make_archive("/kaggle/working/alexnet_results", "zip", OUTPUTS)
elif ON_COLAB:
    archive = shutil.make_archive("/content/alexnet_results", "zip", OUTPUTS)
    print("results zipped to", archive)
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        target = "/content/drive/MyDrive/alexnet_outputs"
        shutil.copytree(OUTPUTS, target, dirs_exist_ok=True)
        print("results copied to Google Drive:", target)
    except Exception as error:
        print("Google Drive not available (", error, "): download", archive, "before the runtime closes")